In [ ]:
import torch
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

In [ ]:
!mkdir infer_workspace

In [ ]:
cd infer_workspace

# inference

In [ ]:
# (marker,path,weight)
# ignore weight here which is used for old infer branch
combs=[
    ('0719_V1','/kaggle/input/datasets/w5833946/rogii-0719-v1-cv509',2.0),
    ('0724_V1','/kaggle/input/datasets/w5833946/rogii-0724-v1-cv486',1.0),
    ('0729_V3','/kaggle/input/datasets/w5833946/rogii-0729-v3-cv553',1.0),
    ('0801_V1','/kaggle/input/datasets/w5833946/rogii-0801-v1-cv516',2.0),
    ('0801_V2','/kaggle/input/datasets/w5833946/rogii-0801-v2-cv480',1.0),
    ('0802_V2','/kaggle/input/datasets/w5833946/rogii-0802-v2-cv513',1.0),
    ('0803_V2','/kaggle/input/datasets/w5833946/rogii-0803-v2-cv500',1.0),
]

In [ ]:
for marker,path,_ in combs:
    source_files=f'{path}/*'
    !cp $source_files ./
    # fix bfloat16 infer bug on kaggle
    cfg=pd.read_pickle('cfg.pkl')
    cfg.amp_dtype='float32'
    pd.to_pickle(cfg,'cfg.pkl')
    # inference
    res_path ='./'
    output_path='/kaggle/working/infer_output'
    data_path='/kaggle/input/competitions/rogii-wellbore-geology-prediction'
    !/usr/local/bin/python seq_NN_main.py\
    --submit-mode\
    --data-dir $data_path \
    --train-output-dir $res_path \
    --output-dir $output_path \
    --device cuda
    # copy infer res
    for f in os.listdir(output_path):
        prefix,postfix=f.split('.')
        from_path=f'{output_path}/{f}'
        to_path=f'/kaggle/working/{prefix}_{marker}.{postfix}'
        !cp $from_path $to_path
    # clean up
    !rm -rf ./*
    !rm -r $output_path

In [ ]:
cd /kaggle/working/

In [ ]:
ls

# ensemble

* use xy_nbr stats from this ver consistently, but it should be same across vers

In [ ]:
xy_stats=pd.read_parquet('/kaggle/input/datasets/w5833946/rogii-0801-v2-cv480/oof_df.pqt')
xy_stats=xy_stats.groupby('well_id')[['geo_nbr_distance','geo_radial_extrap_score', 'geo_nbr_path_alignment', 'geo_nbr_distance_q10','geo_prefix_weight_ratio']].mean()

In [ ]:
xy_stats.columns

In [ ]:
xy_stats.quantile(0.975)

In [ ]:
xy_stats.quantile(0.95)

In [ ]:
xy_stats.quantile(0.025)

In [ ]:
cond=xy_stats['geo_nbr_distance']<2459
cond&=xy_stats['geo_nbr_distance_q10']<1426
cond&=xy_stats['geo_nbr_path_alignment']>0.96
cond&=xy_stats['geo_radial_extrap_score']<1.45 
cond&=xy_stats['geo_prefix_weight_ratio']<0.315
(~cond).sum()

In [ ]:
# GR only means: GR+z_diff, no xy nbr info used
GR_only_combs=[
    ('0719_V1',1.0),
    ('0729_V3',0.5),
    ('0801_V1',1.0),
]
xy_based_combs=[
    ('0719_V1',0.25),
    ('0801_V1',0.25),
    ('0724_V1',1.0),
    ('0801_V2',1.0),
    ('0803_V2',1.0),
]

In [ ]:
for MODE in ['val','test']:
    pred_dfs={}
    for marker,path,_ in combs:
        if MODE=='test':
            pred_dfs[marker]=pd.read_parquet(f'submission_details_{marker}.pqt').sort_values(['well_id','submit_index']).reset_index(drop=True)
        else:
            pred_dfs[marker]=pd.read_parquet(f'{path}/oof_df.pqt').sort_values(['well_id','submit_index']).reset_index(drop=True)
    
    xy_based_markers=list(set([x[0] for x in xy_based_combs])-set([x[0] for x in GR_only_combs]))
    xy_based_marker= '0801_V2' #sorted(xy_based_markers)[-1]
    
    if MODE=='test':
        xy_stats=pd.read_parquet(f'submission_details_{xy_based_marker}.pqt')
    else:
        xy_stats=pd.read_parquet('/kaggle/input/datasets/w5833946/rogii-0801-v2-cv480/oof_df.pqt')
    xy_stats=xy_stats.groupby('well_id')[['geo_nbr_distance','geo_radial_extrap_score', 'geo_nbr_path_alignment', 'geo_nbr_distance_q10','geo_prefix_weight_ratio']].mean()
    
    
    template=pred_dfs[xy_based_marker]
    template['final_pred']=0.0
    for df in pred_dfs.values():
        matched=(df.index==template.index)#.values
        matched&=(df['well_id']==template['well_id']).values
        matched&=(df['submit_index']==template['submit_index']).values
        if matched.mean()!=1:
            print('pred df not matched!!!')
    
    for well_id,sub_df in tqdm(template.groupby('well_id')):
        xy_safe=xy_stats.loc[well_id,'geo_nbr_distance']<2459
        xy_safe&=xy_stats.loc[well_id,'geo_nbr_distance_q10']<1426
        xy_safe&=xy_stats.loc[well_id,'geo_nbr_path_alignment']>0.96
        xy_safe&=xy_stats.loc[well_id,'geo_radial_extrap_score']<1.45
        xy_safe&=xy_stats.loc[well_id,'geo_prefix_weight_ratio']<0.315
        ##xy_safe=True
        if xy_safe:
            used_combs=xy_based_combs
        else:
            used_combs=GR_only_combs
        pred=0.0
        weight_sum=0.0
        for marker,weight in used_combs:
            pred+=pred_dfs[marker].loc[sub_df.index,'TVT_pred'].values*weight
            weight_sum+=weight
        pred/=weight_sum
        template.loc[sub_df.index,'final_pred']=pred
    if MODE=='test':
        template['id']=template['well_id']+'_'+template['submit_index'].astype(str)
        submit_df=template[['id','final_pred']].rename(columns={'final_pred':'tvt'})
        submit_df.to_csv('submission.csv',index=False)
        submit_df
    else:
        rmse=((template['TVT']-template['final_pred'])**2).mean()**0.5
        print(f'val RMSE:{round(rmse,3)}')

In [ ]:
for f in os.listdir('./'):
    if f!='submission.csv':
        !rm -rf $f

In [ ]:
ls

# Reproducing the Training

* All required code is under the dataset path `seq_NN*`, for example:
  `/kaggle/input/datasets/w5833946/rogii-0801-v2-cv480`
* The training configuration used is `cfg.pkl`. You can manually set the same values in `seq_NN_cfg.py` and then start training with `seq_NN_main` using the default arguments.
* In fact, the released code uses the same configuration by default or with `--run-mode seq_full_train`. You can check `SEQ_TRAIN_CFGS` in `seq_NN_cfg.py`. If it contains a key matching the version name (e.g. `date_VX`), then that is the configuration used for training. If no such key exists then it is trained from default cfg without pass `--run-mode seq_full_train`.
* I use a local XY-neighbor clustering file (I appended this to my write-up) to generate the CV folds. However, based on my experiments, it performs very similarly to a simple K-fold split. If you do not have this file, set `cv_split_mode = "kfold"` in `seq_NN_cfg.py`.
* Some path configurations are written for my local environment. You may want to modify them to avoid creating unexpected local directories.
* My local environment uses a single RTX 4080 Super GPU. The Kaggle environment does not support BF16 training, which is required for the ConvNeXt model with BatchNorm.
* I also uploaded my local training log: `seq_nn.log`.
